In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_excel('???? ??? ????.xlsx')

# ?????????

In [ ]:
df.head(5)

In [ ]:
df.iloc[0, 136] = '????? 1-12 ???'

### ??????? NaN ?? ???????? ? ???????? ??????? ??????? ? ???????? ???????

In [ ]:
for i in range(len(df.columns)):
    if type(df.iloc[0, i]) != str:
        df.iloc[0, i] = ''

In [ ]:
for i, column in enumerate(df.columns):
    if 'Unnamed' in column:
        df = df.rename(columns={column: last_column + ' ' + df.iloc[0, i]})
    else:
        last_column = column
        df = df.rename(columns={column: last_column + ' ' + df.iloc[0, i]})

In [ ]:
df.head(5)

In [ ]:
# ?????? ??????? ??????
df = df[1:]

In [ ]:
df.index = [i for i in range(len(df))]

# ???????????? ??????

### ????????? ? ????? ??????? ???????? ?????? ? ???????? ???

In [ ]:
df.columns[174:176]

In [ ]:
not_empty_rows = []
for i, elem in enumerate(zip(df.iloc[:, 174], df.iloc[:, 175])):
    if all([not np.isnan(elem[0]), not np.isnan(elem[1])]):
        not_empty_rows.append(i)

### ???????? ??? ???????, ? ??????? not_empty_rows ??????? ?????? 70% ???????? ????????

In [ ]:
not_empty_columns = []
for column in df.columns:
    try:
        count = 0
        for elem in df[column][not_empty_rows]:
            if np.isnan(elem) == False:
                count += 1
        
        if count/len(not_empty_rows) > 0.7:
            not_empty_columns.append(column)
    except:
        pass

In [ ]:
len(not_empty_columns)

In [ ]:
df_amg = df.loc[:, not_empty_columns]
df_amg = df_amg.iloc[not_empty_rows]

### ?????? ?????? ???????

In [ ]:
for i, column in enumerate(df_amg.columns):
    print(i, column)

In [ ]:
df_amg = df_amg.drop(columns = ['???????, ?????/? (N ?? 6,1; ??>5,6; ??? 6,1-7,0; ?? >7,0) ????? 6-12 ???', '???????, ????/? (2-23) ????? 6-12 ???', '??? 0 - ??? ????? 1,1 ??? ?????; 1- ???????? ????? 1,1 ???? 6-12 ???', '??? 0 - ??? ????? 1,1 ??? ?????; 1- ???????? ????? 1,1 ?? ??', '????? ?????????? ?? ????? >12', '????? ????????? ????? ????? >12', '?????? HOMA-IR (<2,7) (??????? ? ??????? / 22,5) ????? 6-12', '??? ????? ?? ????? 12-24 (?? 2 ???)', '??? ????? 0 - ???, 1 - ????, 4 - ?????????, 5 - ???? (??? ??????? ?? ?????????) ', '?????????? ???????????? ????? ?? 0 -???, 1- ?? ', '??? ????? ?? ????? 12-24 (?? 2 ???)', ])

In [ ]:
df_amg.columns

In [ ]:
df_amg['??????????? ?? ???'] = [1 if elem < 0 else 0 for elem in df_amg['???, ??/?? (0,5-19,6 ??/??) ?? ??'] - df_amg['???, ??/?? (0,5-19,6 ??/??) ???? 6-12 ???']] 

In [ ]:
df_amg.index = [i for i in range(len(df_amg))]

In [ ]:
df_amg = df_amg.drop(columns = ['???, ??/?? (0,5-19,6 ??/??) ???? 6-12 ???'])

In [ ]:
df_amg.head(10)

In [ ]:
feature_scores.index

In [ ]:
feature_scores.index[0:5]

In [ ]:
df_amg = df_amg.loc[:, ['??? ?? ?????? ???????? ', '???? ???, % (<6) ?? ??',
       '?????? HOMA-IR (<2,7) (??????? ? ??????? / 22,5) ?? ??',
       '???, ??/?? (0,5-19,6 ??/??) ?? ??', '??????????? ?? ???']]

In [ ]:
df_amg

# ????????

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import numpy as np
import random
import pickle

In [ ]:
fillna_dict = {column: np.mean(df_amg[column]) for column in df_amg.columns}

In [ ]:
df_amg = df_amg.fillna(fillna_dict)

In [ ]:
X = df_amg.loc[:, df_amg.columns != '??????????? ?? ???']
y = df_amg.loc[:, df_amg.columns == '??????????? ?? ???']

In [ ]:
model_names = ['logreg', 'knn', 'svc', 'nb', 'dt', 'rt', 'xgbx']
models_auc_dict = {model: [] for model in model_names}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    np.array(X.values), np.array(y.values), test_size=0.33, random_state=random.randint(1, 10000))

rt = RandomForestClassifier(n_estimators=100,
                                bootstrap = True,
                                max_features = 'sqrt')
rt.fit(X_train, y_train.ravel())

In [ ]:
pickle.dump(rt, open('model.pkl', 'wb'))

In [ ]:
for i in range(100):
    X_train, X_test, y_train, y_test = train_test_split(
    np.array(X.values), np.array(y.values), test_size=0.33, random_state=random.randint(1, 10000))

    #clf = LogisticRegression(random_state=22, max_iter=1000)
    logreg = LogisticRegression(random_state = 0, max_iter=1000)
    dt = DecisionTreeClassifier()
    knn = KNeighborsClassifier()
    svc = SVC()
    nb = BernoulliNB()
    rt = RandomForestClassifier(n_estimators=100,
                                bootstrap = True,
                                max_features = 'sqrt')
    xgbx = xgb.XGBClassifier()

    models = [logreg, knn, svc, nb, dt, rt, xgbx]

    for model in models:
        model.fit(X_train, y_train.ravel())
    
    for i, model in enumerate(models):
        y_pred = model.predict(X_test)
        
        models_auc_dict[model_names[i]].append(accuracy_score(y_test, y_pred.round(0)))

In [ ]:
for model_name in models_auc_dict.keys():
    print(f'{model_name}: {np.mean(models_auc_dict[model_name])}')

# ???????? ?????????

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
feature_importances = rt.feature_importances_

In [ ]:
for i, column in enumerate(X.columns):
    print(column, feature_importances[i])

In [ ]:
feature_names = X.columns

feature_importances = rt.feature_importances_

feature_scores = pd.Series(rt.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots()
feature_scores[:12].plot.bar(ax=ax)
ax.set_title("Feature importances using MDI (Random Forest)")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

# ????? ??????? ?????????

In [ ]:
from sklearn import decomposition
import matplotlib.pyplot as plt

In [ ]:
# Visualizing using PCA
pca = decomposition.PCA(n_components=4)
res = pca.fit_transform(X)
plt.figure(figsize=(12,8))
plt.scatter(res[:,0], res[:,1], c=y.values, s=50, cmap='viridis')
plt.title('PCA')

In [ ]:
for i, component in enumerate(pca.components_):
    print(
        "{} component: {}% of initial variance".format(
            i + 1, round(100 * pca.explained_variance_ratio_[i], 2)
        )
    )

In [ ]:
plt.scatter(list(X.iloc[:, 4]), list(X.iloc[:, 5]), c=y.values, s=50, cmap='viridis')
plt.title('PCA')